# 11.4 變數作用域：區域變數（Local Scope）與同名變數遮蔽（Shadowing）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_11-4_variable_scope_local_and_shadowing.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**先備知識**：已掌握 11.1 函數定義與呼叫、11.2 函數回傳值，以及 11.3 參數傳遞與副作用防禦。

---

### 學習導覽：探索變數的「活動領地」與「存活壽命」

在撰寫程式碼時，變數並不是在任何地方都可以被隨意讀取或改寫的。每個變數都有其合法的**活動領地（作用域 Scope）**與**存活壽命（生命週期 Lifetime）**。

許多初學者在寫函數時，常會陷入以下困惑：
- 為什麼我在函數裡面算好了一個變數 `ans`，到了函數外面想印出來，電腦卻無情拋出 `NameError: name 'ans' is not defined`？
- 如果外面本來就有一個叫 `total` 的全域變數，函數裡面又定義了一個同名的 `total`，這兩個變數會不會打架打得鼻青臉腫？
- 最詭異的是：為什麼有時候明明只是想把外面的計數器加 1（`count = count + 1`），Python 卻直接引爆 `UnboundLocalError` 讓程式當場崩潰？

在本單元中，我們將以生動形象的比喻，帶你走進變數作用域的底層世界，拆解 6 個關鍵階梯：
1. **11.4.1 什麼是作用域**：認識函數內部的獨立小王國（Local Scope）與邊界防禦。
2. **11.4.2 區域變數的生命週期**：呼叫時誕生、結束時釋放消亡的短暫旅程。
3. **11.4.3 全域與區域同名遮蔽（Shadowing）**：就近原則與「同名黑色遮陽傘」現象。
4. **11.4.4 在函數內唯讀存取全域變數**：LEGB 查找原則與全域常數讀取邊界。
5. **11.4.5 世紀報錯深度剖析**：`UnboundLocalError` 的靜態掃描真相與徹底排查修復。
6. **11.4.6 模組化設計黃金原則**：為什麼專業選手堅持「參數傳入、return 傳出」的純淨解題哲學？

讓我們跟隨清晰的邏輯階梯，徹底搞懂變數作用域，邁向高水準的程式架構思維！

### 11.4.1 什麼是作用域？函數內部的獨立小王國（Local Scope）

#### 1. 生活故事比喻：城堡內部的私人日記
想像主程式是一片遼闊的大陸（全域 Global），而你自訂的每一個函數，就像在這片大陸上建立的一座座「獨立城堡小王國」（區域 Local）。每座城堡都有高聳的城牆。城堡裡面的國王在自己的書房裡寫了一本日記，放在桌上（在函數內宣告了變數 `secret = 123`）。
請問：在大陸平原上散步的外界居民（主程式代碼），能夠隔著厚厚的石牆，伸手看見或拿走那本日記嗎？絕對不行！外界的人如果對著天空大喊：「快把 secret 給我看！」城堡守衛電腦就會立刻亮起紅燈逮捕他，大聲斥責：`NameError: name 'secret' is not defined`（查無此人，這裡沒有這個變數）！

#### 2. 底層運作機制：作用域（Scope）與區域命名空間
在 Python 中，**作用域（Scope）**定義了「變數名稱在哪些程式碼區域內是有效且可見的」：
- **全域作用域（Global Scope）**：在所有函數之外、主程式第一層縮排所定義的變數，屬於全域變數。
- **區域作用域（Local Scope）**：任何在函數內部透過賦值語句（例如 `x = 10`）所建立的變數，或者做為函數參數傳入的變數，都屬於該函數專屬的**區域變數（Local Variable）**。
- **邊界隔離**：函數內部的區域變數，只能在該函數內部使用。函數外部的主程式完全「看不見」它們的存在。

#### 3. 初學者常見陷阱與觀念導正
很多初學同學會寫出這樣的程式碼：
```python
def calc_area(width, height):
    area = width * height  # area 是區域變數

calc_area(10, 20)
print(area)  # 致命錯誤！引發 NameError: name 'area' is not defined
```
初學者常以為：「函數不是剛剛跑過計算出 `area` 了嗎？為什麼不能印？」
請牢記鐵律：**區域變數是函數內部的私人物品，不能直接跨越城牆！**
如果你需要將內部的計算成果交給外界使用，唯一的合法通行證就是使用 **`return area`** 將成果送出，並由外部變數接住！

#### 4. APCS 實戰視野
在 APCS 競賽中，這種「作用域隔離」是保護我們程式碼的最強盾牌！當你為複雜題目拆解出 3 到 5 個輔助函數時，你在每個子函數中隨意命名 `i`、`j`、`temp`、`ans`，完全不必擔心它們會跟其他函數互相干擾打架，因為每個小王國都是徹底獨立安全的。

In [ ]:
# 範例 11.4.1：函數內部區域變數的隔離性與 NameError 示範

def create_secret():
    # secret_code 是函數內部的區域變數
    secret_code = "PASSWORD_999"
    print(f"  [函數內部] 成功存取區域變數: {secret_code}")
    return secret_code

# 主程式執行
print("[主程式] 準備呼叫函數...")
token = create_secret()
print(f"[主程式] 透過 return 成功接收到的數值: {token}")

# 示範如果試圖直接讀取函數內部的區域變數：
# 下面這行若解除註解執行，將引發 NameError: name 'secret_code' is not defined
# print(secret_code)

print("[主程式] 驗證完畢：外部無法直接存取內部的 secret_code！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.4.1
# 任務說明：
# 某位同學想計算三角形面積，並在主程式中取得計算結果。
# 請補齊程式碼中的 `___`：
# 1. 函數內部定義區域變數 triangle_area 計算底乘高除以 2。
# 2. 透過 return 將區域變數的值送出給外部。
# 3. 在主程式宣告變數接收，並印出結果。
# ==========================================

def get_triangle_area(base, height):
    # 提示：計算面積並存入區域變數
    triangle_area = (base * height) / 2
    # 提示：必須透過 return 將區域變數送出
    return ___

# 主程式測試
b, h = 10, 6
# 提示：外部使用變數接住 return 回傳的數值
result_area = get_triangle_area(___, ___)

print(f"三角形面積為: {result_area}")  # 預期輸出: 30.0

In [ ]:
# ==========================================
# [4] Code 練習題 11.4.1
# 任務說明：
# 請設計一個名為 `format_currency(amount)` 的函數：
# 1. 接收金額 amount（整數）。
# 2. 在函數內部建立區域變數 formatted_str，將其格式化為帶有錢幣符號的字串（如 f"${amount:,}" 或 f"${amount}"）。
# 3. 透過 return 回傳該格式化字串。
# 4. 主程式中呼叫兩次此函數，並印出回傳結果。
#
# 【公開測試資料 1】
# 呼叫：format_currency(1250)
# 預期輸出：
# 金額標籤: $1250
#
# 【公開測試資料 2】
# 呼叫：format_currency(88000)
# 預期輸出：
# 金額標籤: $88000
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.4.1
# 任務說明：
# 請設計一個多步驟加密函數 `encode_pin(pin_code)`：
# 1. 接收一個 4 位數純數字字串 pin_code（例如 "1234"）。
# 2. 函數內部使用至少兩個區域變數（如 temp_reversed 存放反轉字串、offset_result 存放數值加總）。
# 3. 計算規則：先將 4 位數字串反轉，再將每個字元轉為整數加 1 後串接成全新字串。
#    （例如 "1234" 反轉為 "4321"，每個數字加 1 得到 "5432"）。
# 4. 函數 return 最終加密字串。
# 5. 主程式驗證輸出，並在註解中說明為什麼外部無法印出函數內的中間變數。
#
# 注意：無公開測試資料，請發揮獨立思維驗證！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.4.2 區域變數的生命週期：呼叫時誕生、結束時釋放消亡

#### 1. 生活故事比喻：舞台上的魔術道具
想像一齣舞台劇中的魔術表演。當導演喊出「Action！」魔術師登場（呼叫函數）時，助手立刻從後台推上道具桌、撲克牌與白鴿（區域變數誕生，電腦配置記憶體空間）。魔術師在台上盡情表演，把撲克牌洗牌、變出鴿子。當這幕戲演完，導演喊出「Cut！」魔術師鞠躬下台（函數執行完畢 return），後台工作人員會毫不留情地將舞台上的所有道具全數清空撤走，一件不留（記憶體回收釋放）。
等到下一場表演再次開演時，舞台是乾乾淨淨的，一切從零開始。前一場表演的撲克牌殘局，絕不可能留在舞台上干擾下一場演出！

#### 2. 底層運作機制：執行堆疊（Call Stack）的壓入與彈出
在電腦底層，區域變數的**生命週期（Lifetime）**非常短暫：
1. **誕生（Allocation）**：每當程式執行到函數呼叫語句（例如 `my_func()`）時，Python 直譯器會在**呼叫堆疊（Call Stack）**上為該函數壓入一個「堆疊框架（Stack Frame）」，並在其中配置專屬的區域命名空間。此時區域變數宣告誕生。
2. **存活（Execution）**：在函數內部的程式碼一行行往下執行期間，區域變數持續存活並保存資料。
3. **消亡（Deallocation）**：一旦函數執行到 `return`，或者執行到最後一行默默結束，該函數的堆疊框架會被立刻**彈出並銷毀**！作業系統與記憶體回收機制（Garbage Collection）會將其中的區域變數全數消滅、釋放記憶體。

#### 3. 初學者的重大迷思：區域變數會「記住上次的數值」嗎？
有些初學者會寫出這樣的程式：
```python
def count_visitors():
    count = 0
    count = count + 1
    print(count)

count_visitors()  # 印出 1
count_visitors()  # 誤以為會印出 2！結果依然是 1！
```
為什麼第二次呼叫依然印出 1？因為第一次呼叫結束時，那個 `count = 1` 早就已經灰飛煙滅了！第二次呼叫是一個全新的輪迴，`count` 再次被初始化為 0，加 1 後依然是 1。**區域變數具有徹底的「失憶性」**。

#### 4. APCS 實戰警語
在 APCS 競賽中，當你在解決多筆測資（Multiple Test Cases）時，將解題邏輯封裝在函數中是非常好的習慣。因為每次呼叫函數解一筆測資，所有區域變數都會乾乾淨淨地重新初始化，絕對不會把上一筆測資的垃圾資料遺留到下一筆測資中，從根源上斬斷了跨測資污染！

In [ ]:
# 範例 11.4.2：驗證區域變數的生命週期與獨立記憶體輪迴

def simulate_round(round_number):
    print(f"--- 第 {round_number} 回合開始 ---")
    # 區域變數誕生
    step_count = 0
    step_count += 5
    print(f"  [函數內部] 本回合累積步數: {step_count}")
    print(f"--- 第 {round_number} 回合結束（step_count 釋放消亡）---\n")

# 連續呼叫三次函數
simulate_round(1)
simulate_round(2)
simulate_round(3)

print("結論：每次呼叫都是獨立的生命週期，區域變數不會跨呼叫累積！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.4.2
# 任務說明：
# 請觀察以下代碼，函數 `greet_guest(guest_name)` 在每次呼叫時建立專屬的迎賓詞。
# 請補齊程式碼中的 `___`，宣告區域變數 message 並回傳，
# 同時在主程式連續呼叫兩次，感受區域變數在不同呼叫間的獨立性。
# ==========================================

def greet_guest(guest_name):
    # 提示：宣告區域變數，組合問候字串
    message = f"歡迎尊貴的賓客：{guest_name}！"
    return ___

# 主程式測試
msg1 = greet_guest("王小明")
msg2 = greet_guest("李小美")

print(msg1)  # 預期輸出: 歡迎尊貴的賓客：王小明！
print(msg2)  # 預期輸出: 歡迎尊貴的賓客：李小美！

In [ ]:
# ==========================================
# [4] Code 練習題 11.4.2
# 任務說明：
# 請設計一個名為 `calc_cart_total(prices, discount)` 的函數：
# 1. prices 為商品價格串列（整數），discount 為折抵現金（整數）。
# 2. 函數內部定義區域變數 subtotal = sum(prices)，
#    再定義 final_total = subtotal - discount（若小於 0 則為 0）。
# 3. 回傳 final_total。
# 4. 在主程式中連續處理兩筆不同的購物車清單，印出各自的最終金額，
#    驗證兩次呼叫間的區域變數完全獨立互不干擾。
#
# 【公開測試資料 1】
# 購物車 1: prices=[100, 200, 300], discount=50
# 預期輸出：
# 購物車 1 結算: 550
#
# 【公開測試資料 2】
# 購物車 2: prices=[50, 40], discount=100
# 預期輸出：
# 購物車 2 結算: 0
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.4.2
# 任務說明：
# 請設計一個生命週期驗證實驗函數 `memory_tracker(tag)`：
# 1. 函數內部建立一個長度為 1000 的區域串列 `temp_list = [0] * 1000`。
# 2. 印出該暫存串列的記憶體編號 `id(temp_list)` 與其標籤 tag。
# 3. 連續呼叫該函數三次。
# 4. 觀察並思考：為什麼不同次呼叫時，temp_list 的 id 有時會相同或相近？
#    （提示：因為前一次呼叫結束時記憶體已被釋放，作業系統重新將剛釋放的地址分配給新呼叫）。
#
# 注意：無公開測試資料，請自行執行並在註解中記錄你的觀察！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.4.3 全域與區域同名：變數遮蔽（Shadowing）現象與就近原則

#### 1. 生活故事比喻：家裡的小貓 vs 鄰居家的小貓
想像你在自己家客廳裡養了一隻可愛的虎斑貓，名字叫「皮皮」（區域變數）。而碰巧，你隔壁鄰居家也養了一隻橘貓，名字也叫「皮皮」（全域變數）。今天你在自己家客廳吃下午茶，溫柔地喊了一聲：「皮皮，過來吃魚乾！」請問，跑過來的是誰？用膝蓋想也知道，絕對是你家客廳的虎斑貓跑過來（就近原則）！隔壁鄰居家的橘貓根本聽不到，更不會隔牆跑進你家吃魚乾。

#### 2. 底層運作機制：就近原則與「同名黑色遮陽傘」
當外部的主程式已經存在一個全域變數 `x = 100`，而某個函數內部又宣告了一個同名變數 `x = 10` 時：
1. **就近原則（Local First）**：當函數內部執行到 `print(x)` 或 `x = x + 1` 時，Python 會優先在該函數自己的區域命名空間（Local Scope）查找。
2. **遮蔽效應（Variable Shadowing）**：因為在區域內找到了同名的 `x`，這個區域變數就像撐起了一把「巨大的黑色遮陽傘」，將外部同名的全域變數擋得嚴嚴實實。
3. **互不相干**：在函數內部對區域變數 `x` 的任何加減乘除或修改，全部都只發生在傘下的內部世界；一旦走出函數回到主程式，那把傘消失了，外面的全域變數 `x` 依然安好地維持著 100，絲毫未損！

#### 3. 變數遮蔽是好是壞？
- **優點（保護性）**：即使外部不小心有一個變數叫 `i` 或 `total`，函數內部的 `for i in range(...)` 也不會把外面的變數改壞，這讓模組化程式具備極佳的健壯性。
- **缺點（閱讀陷阱）**：如果程式設計師原本「誤以為自己正在修改全域變數」，遮蔽現象會讓他以為改成功了，但一出函數才發現全域變數根本沒變，造成邏輯上的假象。

#### 4. APCS 競賽防坑指南
在 APCS 解題時，有些題目會使用全域變數儲存「全域最佳解（如 `ans = 999999`）」。如果你的遞迴或搜尋函數內部，隨手又寫了一句 `ans = min(...)`，此時你就觸發了變數遮蔽，函數內部只是建立了一個同名的臨時區域變數，根本動不到外面的全域最佳解！理解遮蔽機制，是後續 11.5 學習正確修改全域狀態的必備基石。

In [ ]:
# 範例 11.4.3：全域與區域同名變數遮蔽（Shadowing）驗證

# 定義全域變數
hero_level = 99
status = "全域神級英雄"

def enter_dungeon():
    # 宣告同名的區域變數，觸發遮蔽現象
    hero_level = 1
    status = "副本新手初學者"
    print(f"  [副本內部-觸發遮蔽] hero_level = {hero_level}")
    print(f"  [副本內部-觸發遮蔽] status = '{status}'")

print(f"[進入副本前] hero_level = {hero_level}, status = '{status}'")
print("-" * 55)

# 呼叫函數
enter_dungeon()

print("-" * 55)
print(f"[離開副本後] hero_level = {hero_level}, status = '{status}'")
print("結論：內部同名變數只在函數內生效，外部全域變數完好如初！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.4.3
# 任務說明：
# 全域環境中設定了普通商品的稅率 `tax_rate = 0.05`（5%）。
# 某免稅商店函數 `tax_free_shop(price)` 內部定義同名的 `tax_rate = 0.0`。
# 請補齊程式碼中的 `___`，計算免稅後的金額，並在外部驗證全域稅率未被修改。
# ==========================================

tax_rate = 0.05  # 全域稅率

def tax_free_shop(price):
    # 提示：區域變數遮蔽全域變數
    tax_rate = ___  # 免稅商店稅率為 0
    final_price = price * (1 + tax_rate)
    return int(final_price)

# 主程式測試
item_price = 1000
duty_free_price = tax_free_shop(item_price)

print(f"免稅後價格: {duty_free_price}")  # 預期輸出: 1000
print(f"全域稅率依然為: {tax_rate}")     # 預期輸出: 0.05

In [ ]:
# ==========================================
# [4] Code 練習題 11.4.3
# 任務說明：
# 請撰寫程式模擬「班級競賽總分」與「小組內部分數」：
# 1. 全域變數定義 `score = 100`。
# 2. 定義函數 `calc_group_score(base_score, bonus)`：
#    - 內部宣告同名區域變數 `score = base_score + bonus`。
#    - 印出函數內部遮蔽時的 score。
#    - 回傳該區域 score。
# 3. 主程式呼叫此函數兩次，並印出全域 score，證明全域變數保持為 100。
#
# 【公開測試資料 1】
# 呼叫：calc_group_score(50, 20)
# 預期輸出：
# 小組得分: 70
# 全域總分保持: 100
#
# 【公開測試資料 2】
# 呼叫：calc_group_score(80, 15)
# 預期輸出：
# 小組得分: 95
# 全域總分保持: 100
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.4.3
# 任務說明：
# 請設計一個雙層同名遮蔽實驗：
# 1. 定義全域變數 `val = "A (Global)"`。
# 2. 定義函數 `func1()`，內部宣告 `val = "B (Func1)"` 並印出；
#    定義函數 `func2()`，內部宣告 `val = "C (Func2)"` 並印出。
# 3. 依序執行 `func1()`、`func2()`，最後印出全域的 `val`。
# 4. 驗證不同函數內部即使都取名為 `val`，彼此獨立遮蔽，互不干涉。
#
# 注意：無公開測試資料，請自行執行並驗證輸出層級！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.4.4 在函數內部「唯讀」存取全域變數的合法行為與邊界

#### 1. 生活故事比喻：隔著玻璃看展品
想像你走進故宮博物院參觀「翠玉白菜」。展品放在強化玻璃展櫃中（全域變數）。你走近展櫃，仔細端詳白菜上的蟈蟈、朗讀旁邊的解說牌，甚至在自己的小筆記本上記錄它的尺寸大小（純粹讀取 Read-Only）——這一切完全合法，館方非常鼓勵！但是，如果你試圖拿出螺絲起子去撬開玻璃，想把裡面的白菜拿走或換掉（寫入改寫 Write），警報器會立刻尖叫、警衛立刻拔槍！

#### 2. 底層運作機制：Python 的 LEGB 查找規則
當 Python 執行到函數內部的一行代碼，需要取得某個變數的值時，它會按照著名的 **LEGB 順序**由內而外進行地毯式搜索：
1. **L (Local)**：先看當前函數內部有沒有這個區域變數。
2. **E (Enclosing)**：若有外層嵌套函數，看外層函數有沒有（閉包環境）。
3. **G (Global)**：若內部都沒有，走出城堡，看看最外層主程式（全域）有沒有這個變數！
4. **B (Built-in)**：最後看看是不是 Python 內建關鍵字或內建函數（如 `len`, `print`）。

**重要特徵**：如果函數內部**「純粹只是讀取（Read-Only）」**某個全域變數的值（例如做為數學公式的一部分、傳給 print 印出、做為 if 條件判斷），而**完全沒有嘗試對它進行賦值（沒有出現等號賦值）**，那麼 Python 在 Local 找不到時，會非常大方地允許你「借用外面的全域變數」！

#### 3. 適用場景：全域常數（Global Constants）
在良好的程式設計風格中，唯讀讀取全域變數最常見也最推薦的場景，就是**全域常數**。
- 數學常數：`PI = 3.1415926535`
- 演算法大質數模數：`MOD = 1000000007`
- 網格方向向量：`DIRECTIONS = [(0, 1), (1, 0), (0, -1), (-1, 0)]`
將這些不變的常數定義在最上方，讓所有自訂函數以「唯讀」的方式存取，既省去反覆傳遞參數的麻煩，又極其清晰整潔！

#### 4. 嚴禁踩踏的危險邊界
雖然「唯讀」是合法的，但初學者必須隨時保持警戒：**只要你有一絲一毫想要「修改」它的念頭，就會立刻跌入下一個微單元 11.4.5 的世紀大坑！**

In [ ]:
# 範例 11.4.4：在函數內部合法唯讀存取全域常數

# 定義全域常數（慣例使用全大寫英文字母命名）
PI = 3.14159
PLATFORM_NAME = "APCS 官方模擬評判系統"

def calculate_circle_properties(radius):
    # 函數內部並未宣告 PI 或 PLATFORM_NAME，純粹「唯讀存取」全域變數
    circumference = 2 * PI * radius
    area = PI * (radius ** 2)
    print(f"  [系統提示] 本計算由【{PLATFORM_NAME}】自動執行")
    return circumference, area

# 主程式呼叫
r = 10
c, a = calculate_circle_properties(r)

print(f"半徑 {r} 的圓周長: {c:.2f}")
print(f"半徑 {r} 的圓面積: {a:.2f}")
print("確認：函數成功以唯讀方式引用全域常數完成計算！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.4.4
# 任務說明：
# 全域定義了 APCS 及格標準常數 `PASS_STANDARD = 60`。
# 請補齊判定函數 `check_pass(score)` 中的 `___`：
# 函數內部唯讀存取 PASS_STANDARD 進行比較，回傳布林值 True 或 False。
# ==========================================

PASS_STANDARD = 60  # 全域常數

def check_pass(score):
    # 提示：以唯讀方式比對 score 是否大於等於全域常數 PASS_STANDARD
    if score >= ___:
        return True
    else:
        return False

# 主程式測試
print(f"75 分是否及格: {check_pass(75)}")  # 預期: True
print(f"59 分是否及格: {check_pass(59)}")  # 預期: False

In [ ]:
# ==========================================
# [4] Code 練習題 11.4.4
# 任務說明：
# 某外幣兌換中心在全域設定了匯率常數 `USD_TO_TWD = 32.0`。
# 請設計函數 `exchange_to_twd(usd_amount)`：
# 1. 唯讀引用全域常數 USD_TO_TWD，將傳入的美元 usd_amount 換算為新台幣。
# 2. 金額以 int() 取整數回傳。
# 3. 主程式中測試兩組金額，並印出兌換結果。
#
# 【公開測試資料 1】
# 呼叫：exchange_to_twd(50)
# 預期輸出：
# 50 美元兌換新台幣: 1600 元
#
# 【公開測試資料 2】
# 呼叫：exchange_to_twd(125)
# 預期輸出：
# 125 美元兌換新台幣: 4000 元
# ==========================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.4.4
# 任務說明：
# 在全域定義二維網格四方向位移向量常數：
# `DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]`  # 上、下、左、右
# 請設計函數 `get_neighbors(r, c)`：
# 1. 唯讀走訪全域常數 DIRS。
# 2. 計算出 (r, c) 四個相鄰格子的座標元組 (r + dr, c + dc)。
# 3. 回傳包含 4 個座標元組的串列。
# 4. 主程式呼叫 `get_neighbors(5, 5)` 並驗證輸出。
#
# 注意：無公開測試資料，請自行構思驗證四鄰居座標！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.4.5 經典報錯剖析：`UnboundLocalError: local variable referenced before assignment`

#### 1. 生活故事比喻：還沒發號碼牌就想叫號領獎
想像你去一家熱門拉麵店排隊。店員原本有在門口立告示牌說：「今日預設等待人數為 0 人（全域 `counter = 0`）」。當你走進店裡，店員想把號碼牌加一，於是他大喊：「現在號碼牌是幾號？我要給下一號！」可是，店員自己的櫃檯抽屜裡根本還沒放今天的空白號碼簿（區域變數尚未初始化）！這時店員又急著要算「抽屜裡的舊號碼 + 1」，電腦系統直接死機當場抗議：**「你手裡明明沒有區域變數，卻在還沒給它賦值前就想引用它！」**

#### 2. 底層運作機制：Python 的靜態編譯期預先掃描
這是所有 Python 初學者 100% 一定會踩中、且最百思不得其解的世紀大坑！請看這段經典程式碼：
```python
total = 100

def add_bonus():
    print(total)        # 這行看起來很純良對吧？
    total = total + 10  # 關鍵致命行！

add_bonus()
```
初學者直覺想：「既然 11.4.4 說可以唯讀存取全域變數，那第一行 `print(total)` 應該可以印出 100 呀！」
💥 **執行結果**：第一行連印都沒印出來，就直接炸裂報錯：
`UnboundLocalError: local variable 'total' referenced before assignment`！

**為什麼會這樣？底層真相大解密**：
1. Python 在執行一個函數之前，會先進行**編譯期的靜態語法掃描（Compile-time Static Analysis）**。
2. 掃描器一看到函數內部有 `total = ...`（出現了等號賦值），它就立刻鐵面無私地下達判決：
   👉 **「`total` 在這個函數裡是個區域變數（Local Variable）！」**
3. 一旦被打上區域變數標籤，Python 在整個函數內部就**徹底封死了去 Global 查找的通道**！
4. 當真正執行時，第一行 `print(total)` 試圖讀取 `total`，但此時身為區域變數的 `total` 根本還沒有被賦值（它是在下一行才被賦值），因此直譯器判定：**「區域變數尚未賦值即被引用！」**

#### 3. 如何徹底修復此問題？
- **正規軟體架構解法（極力推薦）**：不要試圖直接改寫外部全域變數！將數值作為「參數」傳入，計算完後透過「`return`」回傳：
  ```python
  def add_bonus(val):
      return val + 10
  
  total = add_bonus(total)
  ```
- **競賽特定解法（使用 `global` 關鍵字）**：如果是在 APCS 競賽中為了極限省碼，必須在函數內明確宣告 `global total`（這將在下一單元 11.5 完整傳授）。

#### 4. APCS 考場避坑警示
在 APCS 考試時，當你看到這個錯誤訊息，請不要慌張，立刻檢查你的函數內部是不是同時出現了「讀取」與「賦值」同一個外部變數的名字。把它改成參數傳遞與 return，保證藥到病除！

In [ ]:
# 範例 11.4.5：重現 UnboundLocalError 的成因與正確修復方案

# 【錯誤示範情境（以 try-except 捕捉展示報錯，防崩潰）】
counter = 0

def bad_increment():
    # 只要下面有 counter = ...，整棟函數裡的 counter 就被視為區域變數！
    # 導致這行讀取時引發 UnboundLocalError
    counter = counter + 1

try:
    bad_increment()
except UnboundLocalError as e:
    print("【抓到經典錯誤】:", e)
    print("錯誤原因：Python 靜態掃描發現有賦值，將 counter 標定為 Local，但右側使用時它尚未被賦值！")

print("-" * 60)

# 【正確修復方案：參數進、return 出】
def good_increment(val):
    # 乾淨俐落：透過參數接收、透過 return 傳出
    return val + 1

counter = 0
print(f"呼叫前 counter = {counter}")
counter = good_increment(counter)
print(f"第一次累加後 counter = {counter}")
counter = good_increment(counter)
print(f"第二次累加後 counter = {counter}")
print("修復成功：完全避免作用域衝突！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.4.5
# 任務說明：
# 小傑想寫一個累計積分的函數，原本寫 `user_points = user_points + pts` 引爆錯誤。
# 請補齊程式碼中的 `___`，依照「參數傳入、return 傳出」的最佳實踐重構此函數，
# 讓主程式能夠安全地累計積分。
# ==========================================

# 提示：接收當前積分 current_pts 與欲增加的點數 add_pts
def accumulate_points(current_pts, add_pts):
    # 提示：計算新積分並回傳
    new_pts = current_pts + add_pts
    return ___

# 主程式測試
my_points = 50

# 呼叫函數更新積分
my_points = accumulate_points(my_points, 30)
print(f"更新後積分: {my_points}")  # 預期輸出: 80

my_points = accumulate_points(my_points, 20)
print(f"再次更新後積分: {my_points}")  # 預期輸出: 100

In [ ]:
# ==========================================
# [4] Code 練習題 11.4.5
# 任務說明：
# 請設計一個字串重複追加函數 `append_log(current_log, new_msg)`：
# 1. 接收現有記錄字串 current_log 與新訊息 new_msg。
# 2. 函數內部組裝字串：若 current_log 為空，回傳 new_msg；
#    否則回傳 `f"{current_log} -> {new_msg}"`。
# 3. 透過 return 傳出，主程式中用同一個變數接回，杜絕任何 UnboundLocalError。
# 4. 主程式進行兩輪追加測試並印出最終結果。
#
# 【公開測試資料 1】
# 初始：log = ""
# 依序追加："啟動"、"檢查連線"
# 預期輸出：
# 啟動 -> 檢查連線
#
# 【公開測試資料 2】
# 初始：log = "第一步"
# 依序追加："第二步"、"完工"
# 預期輸出：
# 第一步 -> 第二步 -> 完工
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.4.5
# 任務說明：
# 請寫出一段程式碼，主動觸發一次 `UnboundLocalError`，
# 並在代碼註解中以自己的話（至少 50 字）詳細解釋：
# 1. 為什麼 Python 直譯器會認定該變數是區域變數？
# 2. 為什麼等號右側在讀取時會發生崩潰？
# 隨後，請寫出該段程式碼的「正確重構版」。
#
# 注意：無公開測試資料，請自行設計對照實驗！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

### 11.4.6 設計原則：為什麼盡量依賴「參數傳入、return 傳出」而非依賴外部全域變數？

#### 1. 生活故事比喻：標準規格的樂高積木 vs 纏成死結的毛線球
想像你買了一組高品質的樂高積木。每個積木塊（函數）都有精準的凹槽（參數）與凸起卡榫（return）。你可以隨意把這個積木拿去蓋太空船，也可以拿去蓋中世紀城堡，甚至借給同學組裝，它都能完美契合，因為它只依賴自己的卡榫！
反之，如果一個積木上綁了十幾條透明釣魚線，死死綁在客廳的茶几、沙發與電視機上（緊密依賴外部全域變數）。你只要一把積木拿起來，茶几上的花瓶就被扯翻摔碎、電視機被拽倒。這種程式碼在軟體工程中被稱為**「義大利麵條代碼（Spaghetti Code）」**，是所有工程師的終極惡夢。

#### 2. 核心架構原則：高內聚、低耦合（High Cohesion, Low Coupling）
在現代程式設計中，優秀的函數應具備高度的模組化：
1. **黑盒子封裝（Encapsulation）**：呼叫者只需要知道「傳入什麼參數」以及「會拿到什麼回傳值」，完全不需要知道、也不應該關心函數內部是怎麼實現的。
2. **零副作用（No Side Effects）**：函數執行完畢後，外部世界的其他變數完好如初，沒有莫名其妙被改動。
3. **無可救藥的依賴（Tight Coupling 的危害）**：如果函數依賴某個全域變數 `player_hp`，那麼：
   - 你無法在另一個檔案或另一個題目中重複使用這個函數（因為那裡沒有 `player_hp`）。
   - 你無法輕易為它寫單元測試（Unit Test）。
   - 當程式出錯時，你根本不知道到底是主程式的哪一行把全域變數改壞了，除錯難度呈指數級飆升！

#### 3. APCS 考場的致勝心法：打造隨身「純函數解題工具庫」
在 APCS 競賽中，熟練的選手會在平時就累積自己的純函數工具庫：
- `is_prime(n)`：給定整數，回傳是否為質數。
- `gcd(a, b)`：給定兩數，回傳最大公因數。
- `in_bound(r, c, H, W)`：給定網格座標與長寬，回傳是否在合法範圍內。
這些函數**完全不依賴任何外部全域變數**！只要在考場上一遇到相關題型，立刻默寫貼上，參數丟進去、結果拿出來，100% 穩定、0% 出錯。這就是模組化純函數的極致威力！

In [ ]:
# 範例 11.4.6：緊密耦合的劣質寫法 vs 參數回傳的純函數優雅架構對比

# ❌ 劣質寫法：函數暗中依賴外部全域變數（無法搬遷、除錯極難）
score_list = [80, 90, 70]

def bad_calc_average():
    # 致命隱患：直接存取外部的 score_list，若換了名字整個函數就報廢了！
    return sum(score_list) / len(score_list)

print("劣質寫法計算平均:", bad_calc_average())

print("-" * 55)

# ✅ 優雅架構：純函數（Pure Function），參數傳入、return 傳出
def good_calc_average(numbers):
    """通用平均值計算器：只要傳入任何數值序列皆可精準計算"""
    if not numbers:
        return 0.0
    return sum(numbers) / len(numbers)

# 展現純函數的無限複用彈性：
class_A = [85, 95, 90]
class_B = [60, 70, 80, 90]

print("A 班平均分:", good_calc_average(class_A))
print("B 班平均分:", good_calc_average(class_B))
print("隨手自訂資料:", good_calc_average([100, 100]))
print("結論：純函數具備最高等級的複用性、可讀性與測試便利性！")

In [ ]:
# ==========================================
# [3] Code 填空題 11.4.6
# 任務說明：
# 某位同學原本寫了一個依賴全域變數 `w` 與 `h` 的周長計算函數。
# 請補齊程式碼中的 `___`，將其重構為標準純函數 `calc_perimeter(width, height)`，
# 讓該函數可以傳入任意長寬進行計算並回傳。
# ==========================================

# 提示：定義兩個位置參數接收長與寬
def calc_perimeter(___, ___):
    perimeter = 2 * (width + height)
    return ___

# 主程式測試
rect1_p = calc_perimeter(10, 5)
rect2_p = calc_perimeter(20, 8)

print(f"矩形 1 周長: {rect1_p}")  # 預期輸出: 30
print(f"矩形 2 周長: {rect2_p}")  # 預期輸出: 56

In [ ]:
# ==========================================
# [4] Code 練習題 11.4.6
# 任務說明：
# 請設計一個通用曼哈頓距離計算函數 `manhattan_distance(p1, p2)`：
# 1. p1 與 p2 各為包含兩個整數座標的元組 (x, y)，例如 p1=(x1, y1), p2=(x2, y2)。
# 2. 函數內部完全不依賴外部全域變數，計算公式為 abs(x1 - x2) + abs(y1 - y2)。
# 3. 透過 return 回傳整數距離。
# 4. 主程式測試兩組點對距離並印出。
#
# 【公開測試資料 1】
# 呼叫：manhattan_distance((0, 0), (3, 4))
# 預期輸出：
# 點 (0, 0) 與 (3, 4) 的曼哈頓距離: 7
#
# 【公開測試資料 2】
# 呼叫：manhattan_distance((10, 20), (15, 8))
# 預期輸出：
# 點 (10, 20) 與 (15, 8) 的曼哈頓距離: 17
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

In [ ]:
# ==========================================
# [5] Code 挑戰題 11.4.6
# 任務說明：
# 請打造一個標準純函數解題小工具 `is_leap_year(year)`：
# 1. 接收西元年份 year（整數）。
# 2. 判斷閏年規則：若 year 能被 400 整除為閏年（True）；
#    若能被 100 整除則為平年（False）；若能被 4 整除為閏年（True）；其餘為平年（False）。
# 3. 函數內部完全封裝，回傳布林值 True 或 False。
# 4. 主程式呼叫測試多個年份（如 2000, 1900, 2024, 2023）並印出驗證。
#
# 注意：無公開測試資料，請自行發揮獨立思維驗證！
# ==========================================
# 請在下方撰寫你的程式碼並執行測試：

## 11.4 單元重點回顧與自我檢驗

恭喜你完成第 11.4 單元的深度修練！變數作用域與生命週期是程式架構中最核心的骨架觀念。理解了函數內部的獨立小王國與 LEGB 規則，你已經擁有了看穿程式變數行為的「透視眼」。

### 核心觀念複習清單
1. **區域作用域（Local Scope）**：
   - 函數內宣告的變數與參數屬於 Local Scope，外部主程式絕對無法直接存取，硬要存取引發 `NameError`。
   - 想要交接計算成果，必須依賴 `return`。
2. **生命週期（Lifetime）**：
   - 區域變數在函數被「呼叫」時誕生，在函數執行結束時立刻消亡釋放。
   - 每次呼叫都是獨立乾淨的記憶體輪迴，絕不會跨呼叫殘留記憶。
3. **同名變數遮蔽（Shadowing）**：
   - 當區域變數與全域變數同名時，依據「就近原則」，函數內部優先使用區域變數。
   - 區域變數像一把黑傘遮蔽了全域變數，內部操作絲毫不影響外面的全域變數。
4. **唯讀存取全域變數（LEGB 規則）**：
   - 若函數內部純粹讀取某全域變數且從未對其賦值，Python 會沿著 Local $ightarrow$ Global 順序找到它並合法讀取（適用於全域常數）。
5. **世紀報錯 `UnboundLocalError` 排查**：
   - 函數只要內部出現過 `x = ...`，Python 編譯期靜態掃描就認定 `x` 是 Local。
   - 若在賦值前就嘗試讀取 `x`，就會引發崩潰。
   - 最佳解法：改用「參數傳入、return 傳出」！
6. **模組化設計哲學**：
   - 堅持純函數（Pure Function），高內聚、低耦合，打造即插即用的強大解題工具庫。

---

### 下一步精彩預告
既然我們知道了在函數內部隨便修改全域變數會引發 `UnboundLocalError`，那麼：
- 如果在 APCS 競賽中，我真的非常需要維護一個「全地圖最佳解計數器」，或者共用一個高達 1000x1000 的龐大地圖陣列，該怎麼辦？
- 著名的 `global` 關鍵字究竟是如何合法打破作用域屏障的？
- 競賽中使用 `global` 有什麼致命陷阱？為什麼許多選手會因為沒寫重置（Reset）而慘遭 0 分？

請緊接著邁向 **[11.5 全域變數（global）在 APCS 競賽中的使用準則與除錯防禦]**，解鎖競賽實戰黑科技！